In [1]:
import os
import pickle
import pandas as pd
import numpy as np

# =========================
# Paths and settings
# =========================
base_dir = r"D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results"

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]

participants = [str(p) for p in participants]

samplers = ['CMA-ES', 'TPE']

# =========================
# Read pkl files and build table
# =========================
rows = []

for sampler in samplers:
    for participant in participants:
        folder = os.path.join(base_dir, sampler, participant)

        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".pkl"):
                continue

            fpath = os.path.join(folder, fname)

            try:
                with open(fpath, "rb") as f:
                    data = pickle.load(f)
            except Exception as e:
                print(f"Could not read {fpath}: {e}")
                continue

            metadata = data.get("metadata", {})
            best_params = data.get("best_params", {})
            lapse_rates = data.get("lapse_rates", {})

            # Try possible names for test/CV error
            test_error = np.nan
            for key in ["test_error", "cv_error", "validation_error", "val_error"]:
                if key in data:
                    test_error = data[key]
                    break

            participant_id = str(metadata.get("Participant_ID", participant))
            sampler_name = str(metadata.get("sampler", sampler))

            row = {
                "participant": participant_id,
                "Sampler": sampler_name,
                "train_error": data.get("best_error", np.nan),
                "test_error": test_error,
                "source_file": fname,
            }

            # Add model parameters
            for param_name, param_value in best_params.items():
                row[param_name] = param_value

            # Add lapse rates
            row["lambda_A"] = lapse_rates.get("lambda_A", np.nan)
            row["lambda_B"] = lapse_rates.get("lambda_B", np.nan)

            rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(
        "No .pkl files were found or loaded. Check base_dir, sampler folders, and participant IDs."
    )

# =========================
# Clean column order
# =========================
preferred_order = [
    "participant",
    "Sampler",
    "sigma_noise",
    "A_repulsion",
    "gamma",
    "sigma_update",
    "eta_learning",
    "sigma_boundary",
    "alpha",
    "lambda_A",
    "lambda_B",
    "train_error",
    "test_error",
    "source_file",
]

existing_cols = [c for c in preferred_order if c in df.columns]
remaining_cols = [c for c in df.columns if c not in existing_cols]
df = df[existing_cols + remaining_cols]

# =========================
# Save CSV
# =========================
output_csv = os.path.join(base_dir, "all_fit_results_summary.csv")
df.to_csv(output_csv, index=False)

print(f"Saved CSV to:\n{output_csv}")
print(df.head())

Saved CSV to:
D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\all_fit_results_summary.csv
  participant Sampler  sigma_noise  A_repulsion     gamma  sigma_update  \
0     6363339  CMA-ES     0.131380     0.090902  0.577314      0.605562   
1     6363334  CMA-ES     0.029888     0.493619  0.618233      0.492842   
2     6363353  CMA-ES     0.060992     0.376696  0.523984      0.171761   
3     6363338  CMA-ES     0.022399     0.045033  0.447703      0.375857   
4     6363346  CMA-ES     0.102165     0.144214  0.569037      0.330332   

   eta_learning  sigma_boundary  alpha  lambda_A  lambda_B  train_error  \
0      0.065217        5.427195    0.0  0.126427  0.060508     0.015301   
1      0.130392        5.720072    0.0  0.005453  0.000000     0.019707   
2      0.037968        5.112113    0.0  0.000000  0.005643     0.007039   
3      0.228005        8.275798    0.0  0.180523  0.126541     0.052569   
4      0.187715        8.299070    0.0  0.058229  0.108308     0.026719   

  

In [2]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================
# Paths and settings
# =========================
base_dir = r"D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results"
output_dir = os.path.join(base_dir, "summary_outputs")
os.makedirs(output_dir, exist_ok=True)

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]

participants = [str(p) for p in participants]

samplers = ['CMA-ES', 'TPE']

# Use only a few clean markers.
# They are assigned with a spacing trick so similar nearby participants are less likely
# to get the same marker.
good_markers = ['o', 's', '^', 'D', 'v', 'P', 'X']

participant_markers = {}
for i, p in enumerate(participants):
    marker_index = (i * 3) % len(good_markers)
    participant_markers[p] = good_markers[marker_index]

# Color for each sampler
sampler_colors = {
    'CMA-ES': 'blue',
    'TPE': 'red',
}

# =========================
# Read pkl files and build table
# =========================
rows = []

for sampler in samplers:
    for participant in participants:
        folder = os.path.join(base_dir, sampler, participant)

        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".pkl"):
                continue

            fpath = os.path.join(folder, fname)

            try:
                with open(fpath, "rb") as f:
                    data = pickle.load(f)
            except Exception as e:
                print(f"Could not read {fpath}: {e}")
                continue

            metadata = data.get("metadata", {})
            best_params = data.get("best_params", {})
            lapse_rates = data.get("lapse_rates", {})

            # Try a few possible names for test error
            test_error = np.nan
            for key in ["test_error", "cv_error", "validation_error", "val_error"]:
                if key in data:
                    test_error = data[key]
                    break

            participant_id = str(metadata.get("Participant_ID", participant))
            sampler_name = str(metadata.get("sampler", sampler))

            row = {
                "participant": participant_id,
                "Sampler": sampler_name,
                "train_error": data.get("best_error", np.nan),
                "test_error": test_error,
            }

            # Add fitted parameters
            for k, v in best_params.items():
                row[k] = v

            # Add lapse rates
            row["lambda_A"] = lapse_rates.get("lambda_A", np.nan)
            row["lambda_B"] = lapse_rates.get("lambda_B", np.nan)

            rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(
        "No .pkl files were found or loaded. Check base_dir, sampler folders, and participant IDs."
    )

# =========================
# Clean column order if present
# =========================
preferred_order = [
    "participant", "Sampler",
    "sigma_noise", "A_repulsion", "gamma", "sigma_update",
    "eta_learning", "sigma_boundary", "alpha",
    "lambda_A", "lambda_B",
    "train_error", "test_error"
]

existing_cols = [c for c in preferred_order if c in df.columns]
remaining_cols = [c for c in df.columns if c not in existing_cols]
df = df[existing_cols + remaining_cols]

# Keep this for later, but do not save CSV for now
# csv_path = os.path.join(output_dir, "all_fit_results_summary.csv")
# df.to_csv(csv_path, index=False)
# print(f"CSV saved to: {csv_path}")

# =========================
# Plotting helpers
# =========================
def make_sampler_legend():
    sampler_handles = [
        Line2D(
            [0], [0],
            marker='o',
            color='blue',
            linestyle='None',
            markersize=9,
            label='CMA-ES'
        ),
        Line2D(
            [0], [0],
            marker='o',
            color='red',
            linestyle='None',
            markersize=9,
            label='TPE'
        )
    ]

    return sampler_handles


def plot_single_parameter(df, param_name, save_path):
    fig, ax = plt.subplots(figsize=(12, 6))

    x_positions = {p: i for i, p in enumerate(participants)}

    for _, row in df.iterrows():
        participant = str(row["participant"])
        sampler = str(row["Sampler"])

        if participant not in x_positions:
            continue

        if param_name not in row or pd.isna(row[param_name]):
            continue

        x = x_positions[participant]

        # Slight offset so CMA-ES and TPE do not overlap exactly
        offset = -0.12 if sampler == "CMA-ES" else 0.12

        ax.scatter(
            x + offset,
            row[param_name],
            marker=participant_markers.get(participant, 'o'),
            color=sampler_colors.get(sampler, 'black'),
            s=110,
            alpha=0.9
        )

    ax.set_xticks(range(len(participants)))
    ax.set_xticklabels(participants, rotation=45, ha="right")
    ax.set_ylabel(param_name)
    ax.set_title(param_name)
    ax.grid(True, alpha=0.3)

    ax.legend(
        handles=make_sampler_legend(),
        title="Sampler",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0)
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")


def plot_lambda_together(df, save_path):
    fig, ax = plt.subplots(figsize=(12, 6))

    base_positions = {p: i for i, p in enumerate(participants)}

    for _, row in df.iterrows():
        participant = str(row["participant"])
        sampler = str(row["Sampler"])

        if participant not in base_positions:
            continue

        base_x = base_positions[participant]

        # Sampler offset
        sampler_offset = -0.15 if sampler == "CMA-ES" else 0.15

        # lambda_A a bit left, lambda_B a bit right
        if "lambda_A" in row and not pd.isna(row["lambda_A"]):
            ax.scatter(
                base_x - 0.05 + sampler_offset,
                row["lambda_A"],
                marker=participant_markers.get(participant, 'o'),
                color=sampler_colors.get(sampler, 'black'),
                s=110,
                alpha=0.9
            )

        if "lambda_B" in row and not pd.isna(row["lambda_B"]):
            ax.scatter(
                base_x + 0.05 + sampler_offset,
                row["lambda_B"],
                marker=participant_markers.get(participant, 'o'),
                color=sampler_colors.get(sampler, 'black'),
                s=110,
                alpha=0.9
            )

    ax.set_xticks(range(len(participants)))
    ax.set_xticklabels(participants, rotation=45, ha="right")
    ax.set_ylabel("Value")
    ax.set_title("lambda_A and lambda_B")
    ax.grid(True, alpha=0.3)

    lambda_handles = [
        Line2D(
            [0], [0],
            marker='o',
            color='gray',
            linestyle='None',
            markersize=8,
            label='Left point: lambda_A'
        ),
        Line2D(
            [0], [0],
            marker='o',
            color='gray',
            linestyle='None',
            markersize=8,
            label='Right point: lambda_B'
        )
    ]

    legend1 = ax.legend(
        handles=make_sampler_legend(),
        title="Sampler",
        loc="upper left",
        bbox_to_anchor=(1.02, 1.0)
    )
    ax.add_artist(legend1)

    ax.legend(
        handles=lambda_handles,
        title="Lambda",
        loc="upper left",
        bbox_to_anchor=(1.02, 0.75)
    )

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")


# =========================
# Make plots
# =========================
single_plot_params = [
    c for c in [
        "sigma_noise", "A_repulsion", "gamma", "sigma_update",
        "eta_learning", "sigma_boundary", "alpha",
        "train_error", "test_error"
    ]
    if c in df.columns
]

for param in single_plot_params:
    plot_path = os.path.join(output_dir, f"{param}.png")
    plot_single_parameter(df, param, plot_path)

# lambda_A and lambda_B in the same figure
if "lambda_A" in df.columns or "lambda_B" in df.columns:
    plot_lambda_together(
        df,
        os.path.join(output_dir, "lambda_A_lambda_B.png")
    )

print("Done.")

Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\sigma_noise.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\A_repulsion.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\gamma.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\sigma_update.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\eta_learning.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\sigma_boundary.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\alpha.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\train_error.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\test_error.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs\lambda_A_lambda_B.png
Done.


In [3]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================
# Paths and settings
# =========================
base_dir = r"D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results"
output_dir = os.path.join(base_dir, "summary_outputs_per_participant")
os.makedirs(output_dir, exist_ok=True)

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]

participants = [str(p) for p in participants]

samplers = ['CMA-ES', 'TPE']

sampler_colors = {
    'CMA-ES': 'blue',
    'TPE': 'red',
}

sampler_markers = {
    'CMA-ES': 'o',
    'TPE': 'D',
}

# =========================
# Read pkl files and build table
# =========================
rows = []

for sampler in samplers:
    for participant in participants:
        folder = os.path.join(base_dir, sampler, participant)

        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".pkl"):
                continue

            fpath = os.path.join(folder, fname)

            try:
                with open(fpath, "rb") as f:
                    data = pickle.load(f)
            except Exception as e:
                print(f"Could not read {fpath}: {e}")
                continue

            metadata = data.get("metadata", {})
            best_params = data.get("best_params", {})
            lapse_rates = data.get("lapse_rates", {})

            test_error = np.nan
            for key in ["test_error", "cv_error", "validation_error", "val_error"]:
                if key in data:
                    test_error = data[key]
                    break

            participant_id = str(metadata.get("Participant_ID", participant))
            sampler_name = str(metadata.get("sampler", sampler))

            row = {
                "participant": participant_id,
                "Sampler": sampler_name,
                "train_error": data.get("best_error", np.nan),
                "test_error": test_error,
                "source_file": fname,
            }

            for k, v in best_params.items():
                row[k] = v

            row["lambda_A"] = lapse_rates.get("lambda_A", np.nan)
            row["lambda_B"] = lapse_rates.get("lambda_B", np.nan)

            rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(
        "No .pkl files were found or loaded. Check base_dir, sampler folders, and participant IDs."
    )

# Save csv too
# csv_path = os.path.join(output_dir, "all_fit_results_summary.csv")
# df.to_csv(csv_path, index=False)
# print(f"CSV saved to: {csv_path}")

# =========================
# Choose columns for x-axis
# =========================
preferred_param_order = [
    "sigma_noise",
    "A_repulsion",
    "gamma",
    "sigma_update",
    "eta_learning",
    "sigma_boundary",
    "alpha",
    "lambda_A",
    "lambda_B",
    "train_error",
    "test_error",
]

param_columns = [c for c in preferred_param_order if c in df.columns]

if len(param_columns) == 0:
    raise RuntimeError("No parameter columns found to plot.")

# =========================
# Plot one figure per participant
# =========================
for participant in participants:
    subdf = df[df["participant"].astype(str) == participant].copy()

    if subdf.empty:
        print(f"No data for participant {participant}")
        continue

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(param_columns))

    for sampler in samplers:
        row_sampler = subdf[subdf["Sampler"].astype(str) == sampler]

        if row_sampler.empty:
            continue

        # If multiple rows exist for the same participant/sampler, keep the best one
        if "train_error" in row_sampler.columns:
            row_sampler = row_sampler.sort_values("train_error", ascending=True)

        row = row_sampler.iloc[0]

        y = [row.get(param, np.nan) for param in param_columns]

        ax.scatter(
            x,
            y,
            marker=sampler_markers.get(sampler, 'o'),
            color=sampler_colors.get(sampler, 'black'),
            s=90,
            alpha=0.9,
            label=sampler
        )

    ax.set_xticks(x)
    ax.set_xticklabels(param_columns, rotation=45, ha='right')
    ax.set_ylabel("Value")
    ax.set_title(f"Participant {participant}")
    ax.grid(True, alpha=0.3)

    legend_handles = [
        Line2D(
            [0], [0],
            marker='o',
            color='blue',
            linestyle='None',
            markersize=8,
            label='CMA-ES'
        ),
        Line2D(
            [0], [0],
            marker='D',
            color='red',
            linestyle='None',
            markersize=8,
            label='TPE'
        ),
    ]

    ax.legend(handles=legend_handles, title="Sampler")

    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{participant}_parameter_profile.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")

print("Done.")

Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363339_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363334_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363353_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363338_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363346_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363343_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\6363329_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant\636335

In [4]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# =========================
# Paths and settings
# =========================
base_dir = r"D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results"
output_dir = os.path.join(base_dir, "summary_outputs_per_participant2")
os.makedirs(output_dir, exist_ok=True)

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]

participants = [str(p) for p in participants]

samplers = ['CMA-ES', 'TPE']

sampler_colors = {
    'CMA-ES': 'blue',
    'TPE': 'red',
}

sampler_markers = {
    'CMA-ES': 'o',
    'TPE': 'D',
}

# =========================
# Read pkl files
# =========================
rows = []

for sampler in samplers:
    for participant in participants:
        folder = os.path.join(base_dir, sampler, participant)

        if not os.path.isdir(folder):
            print(f"Folder not found: {folder}")
            continue

        for fname in os.listdir(folder):
            if not fname.endswith(".pkl"):
                continue

            fpath = os.path.join(folder, fname)

            try:
                with open(fpath, "rb") as f:
                    data = pickle.load(f)
            except Exception as e:
                print(f"Could not read {fpath}: {e}")
                continue

            metadata = data.get("metadata", {})
            best_params = data.get("best_params", {})
            lapse_rates = data.get("lapse_rates", {})

            participant_id = str(metadata.get("Participant_ID", participant))
            sampler_name = str(metadata.get("sampler", sampler))

            row = {
                "participant": participant_id,
                "Sampler": sampler_name,
                "train_error": data.get("best_error", np.nan),
                "source_file": fname,
            }

            for k, v in best_params.items():
                row[k] = v

            row["lambda_A"] = lapse_rates.get("lambda_A", np.nan)
            row["lambda_B"] = lapse_rates.get("lambda_B", np.nan)

            rows.append(row)

df = pd.DataFrame(rows)

if df.empty:
    raise RuntimeError(
        "No .pkl files were found or loaded. Check base_dir, sampler folders, and participant IDs."
    )

# =========================
# Columns to plot
# EXCLUDE sigma_boundary and test_error
# =========================
param_columns = [
    "sigma_noise",
    "A_repulsion",
    "gamma",
    "sigma_update",
    "eta_learning",
    "alpha",
    "lambda_A",
    "lambda_B",
    "train_error",
]

param_columns = [c for c in param_columns if c in df.columns]

if len(param_columns) == 0:
    raise RuntimeError("No parameter columns found to plot.")

# =========================
# Plot per participant
# =========================
for participant in participants:
    subdf = df[df["participant"].astype(str) == participant].copy()

    if subdf.empty:
        print(f"No data for participant {participant}")
        continue

    fig, ax = plt.subplots(figsize=(12, 6))

    x = np.arange(len(param_columns))

    for sampler in samplers:
        row_sampler = subdf[subdf["Sampler"].astype(str) == sampler].copy()

        if row_sampler.empty:
            continue

        # If multiple rows exist for the same participant/sampler, keep the best one
        if "train_error" in row_sampler.columns:
            row_sampler = row_sampler.sort_values("train_error", ascending=True)

        row = row_sampler.iloc[0]
        y = [row.get(param, np.nan) for param in param_columns]

        ax.scatter(
            x,
            y,
            marker=sampler_markers.get(sampler, 'o'),
            color=sampler_colors.get(sampler, 'black'),
            s=100,
            alpha=0.9,
            label=sampler
        )

    ax.set_xticks(x)
    ax.set_xticklabels(param_columns, rotation=45, ha='right')
    ax.set_ylabel("Value")
    ax.set_title(f"Participant {participant}")
    ax.grid(True, alpha=0.3)

    legend_handles = [
        Line2D(
            [0], [0],
            marker='o',
            color='blue',
            linestyle='None',
            markersize=8,
            label='CMA-ES'
        ),
        Line2D(
            [0], [0],
            marker='D',
            color='red',
            linestyle='None',
            markersize=8,
            label='TPE'
        ),
    ]

    ax.legend(handles=legend_handles, title="Sampler")

    plt.tight_layout()

    save_path = os.path.join(output_dir, f"{participant}_parameter_profile.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")

print("Done.")

Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363339_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363334_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363353_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363338_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363346_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363343_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant2\6363329_parameter_profile.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\summary_outputs_per_participant

In [ ]:
import os
import pickle
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# =========================
# Paths and settings
# =========================
base_dir = r"D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results"
output_dir = os.path.join(base_dir, "matrix_plots_per_participant")
os.makedirs(output_dir, exist_ok=True)

participants = [
    '6363339', '6363334', '6363353', '6363338', '6363346', '6363343',
    '6363329', '6363354', '6363347', '6363328', '6363330', '6363341',
    '6363407', '6363430', '6363368', '6363349', '6363364', '6269773',
    '6269770', '6269757', '6269881', '6357507', '6357495', '6357501',
    '6357497', '6357556', '6357698', '6357829'
]

participants = [str(p) for p in participants]

samplers = ['CMA-ES', 'TPE']

# =========================
# Colormap and normalization
# =========================
colors = ["orange", "white", "purple"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
norm_model = TwoSlopeNorm(vmin=-0.2, vcenter=0, vmax=0.2)

# =========================
# Matrix keys
# =========================
matrix_keys = [
    ("target_update", "Empirical"),
    ("sim_Final_update", "Hybrid"),
    ("sim_SC_update", "SC"),
    ("sim_BE_update", "BE"),
]

extent = [-1, 1, -1, 1]
tick_positions = np.linspace(-1, 1, 9)
tick_labels = [-1, -0.75, -0.5, -0.25, 0, 0.25, 0.5, 0.75, 1]

# =========================
# Helpers
# =========================
def load_pickle(path):
    with open(path, "rb") as f:
        return pickle.load(f)


def find_participant_file(base_dir, sampler, participant):
    participant = str(participant)
    folder = os.path.join(base_dir, sampler, participant)

    if not os.path.isdir(folder):
        print(f"Folder not found: {folder}")
        return None

    pkl_files = [f for f in os.listdir(folder) if f.endswith(".pkl")]

    if not pkl_files:
        print(f"No .pkl files found in: {folder}")
        return None

    expected_name = f"results_pid_{participant}_seed_1_{sampler}.pkl"

    if expected_name in pkl_files:
        return os.path.join(folder, expected_name)

    # If the expected file does not exist, choose the first file alphabetically
    pkl_files = sorted(pkl_files)
    return os.path.join(folder, pkl_files[0])


def get_matrix(data, key):
    mat = data.get("final_matrices", {}).get(key, None)

    if mat is None:
        mat = data.get(key, None)

    if mat is None:
        return None

    mat = np.array(mat, dtype=float)

    if mat.ndim != 2:
        print(f"Skipping {key}: expected 2D matrix, got shape {mat.shape}")
        return None

    return mat


def get_alpha_percent_titles(data):
    alpha = data.get("best_params", {}).get("alpha", np.nan)

    try:
        alpha = float(alpha)
    except Exception:
        alpha = np.nan

    if np.isnan(alpha):
        return "SC", "BE"

    alpha = np.clip(alpha, 0, 1)
    sc_percent = int(round(alpha * 100))
    be_percent = 100 - sc_percent

    sc_title = f"SC ({sc_percent}%)"
    be_title = f"BE ({be_percent}%)"

    return sc_title, be_title


def style_axis(ax, row_idx, col_idx, title, sampler_label=None):
    ax.set_xticks(tick_positions)
    ax.set_xticklabels(tick_labels, fontsize=8)
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(tick_labels, fontsize=8)
    ax.set_title(title, fontsize=10)

    if col_idx != 0:
        ax.set_yticklabels([])

    if row_idx == 0:
        ax.set_xticklabels([])

    if sampler_label is not None and col_idx == 0:
        ax.set_ylabel(sampler_label, fontsize=12)


def plot_grid(data_by_sampler, figure_title, save_path):
    fig, axes = plt.subplots(
        len(samplers),
        len(matrix_keys),
        figsize=(16, 8),
        dpi=300,
        constrained_layout=True
    )

    if len(samplers) == 1:
        axes = np.expand_dims(axes, axis=0)

    last_im = None

    for row_idx, sampler in enumerate(samplers):
        data = data_by_sampler.get(sampler, None)

        if data is None:
            for col_idx, (_, base_title) in enumerate(matrix_keys):
                ax = axes[row_idx, col_idx]
                ax.axis("off")
                ax.set_title(base_title, fontsize=10)
            continue

        sc_title, be_title = get_alpha_percent_titles(data)

        titles_for_this_row = [
            "Empirical",
            "Hybrid",
            sc_title,
            be_title,
        ]

        for col_idx, ((key, _), title) in enumerate(zip(matrix_keys, titles_for_this_row)):
            ax = axes[row_idx, col_idx]
            mat = get_matrix(data, key)

            if mat is None:
                ax.axis("off")
                ax.set_title(title, fontsize=10)
                continue

            im = ax.imshow(
                mat,
                cmap=cmap,
                norm=norm_model,
                aspect='equal',
                interpolation='nearest',
                origin='upper',
                extent=extent
            )

            last_im = im
            style_axis(ax, row_idx, col_idx, title, sampler_label=sampler)

    fig.supxlabel("Previous stimulus", fontsize=14)
    fig.supylabel("Current stimulus", fontsize=14)

    if last_im is not None:
        cbar = fig.colorbar(last_im, ax=axes.ravel().tolist(), shrink=0.8)
        cbar.set_label("Update strength", fontsize=10)

    fig.suptitle(figure_title, fontsize=16)

    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved plot: {save_path}")


# =========================
# Per-participant plots
# =========================
for participant in participants:
    data_by_sampler = {}

    for sampler in samplers:
        fpath = find_participant_file(base_dir, sampler, participant)

        if fpath is None:
            data_by_sampler[sampler] = None
            continue

        try:
            data_by_sampler[sampler] = load_pickle(fpath)
        except Exception as e:
            print(f"Could not read {fpath}: {e}")
            data_by_sampler[sampler] = None

    if all(data_by_sampler.get(s) is None for s in samplers):
        print(f"No files found for participant {participant}")
        continue

    save_path = os.path.join(output_dir, f"{participant}_update_matrices.png")

    plot_grid(
        data_by_sampler=data_by_sampler,
        figure_title=f"Participant {participant}",
        save_path=save_path
    )

# =========================
# Mean-over-participants plot
# =========================
mean_data_by_sampler = {}

for sampler in samplers:
    collected = {
        "target_update": [],
        "sim_Final_update": [],
        "sim_SC_update": [],
        "sim_BE_update": [],
        "alpha": []
    }

    for participant in participants:
        fpath = find_participant_file(base_dir, sampler, participant)

        if fpath is None:
            continue

        try:
            data = load_pickle(fpath)
        except Exception as e:
            print(f"Could not read {fpath}: {e}")
            continue

        for key, _ in matrix_keys:
            mat = get_matrix(data, key)

            if mat is not None:
                collected[key].append(mat)

        alpha = data.get("best_params", {}).get("alpha", np.nan)

        try:
            alpha = float(alpha)
        except Exception:
            alpha = np.nan

        if not np.isnan(alpha):
            collected["alpha"].append(alpha)

    if all(len(collected[key]) == 0 for key, _ in matrix_keys):
        mean_data_by_sampler[sampler] = None
        continue

    mean_final_matrices = {}

    for key, _ in matrix_keys:
        if len(collected[key]) > 0:
            shapes = [m.shape for m in collected[key]]

            if len(set(shapes)) != 1:
                print(f"Skipping mean for {sampler}, {key}: inconsistent shapes {set(shapes)}")
                continue

            mean_final_matrices[key] = np.mean(
                np.stack(collected[key], axis=0),
                axis=0
            )

    mean_alpha = np.nan

    if len(collected["alpha"]) > 0:
        mean_alpha = np.mean(collected["alpha"])

    mean_data_by_sampler[sampler] = {
        "final_matrices": mean_final_matrices,
        "best_params": {"alpha": mean_alpha}
    }

mean_save_path = os.path.join(output_dir, "MEAN_update_matrices.png")

plot_grid(
    data_by_sampler=mean_data_by_sampler,
    figure_title="Mean over participants",
    save_path=mean_save_path
)

print("Done.")

Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363339_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363334_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363353_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363338_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363346_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363343_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363329_update_matrices.png
Saved plot: D:\sc_modelling-amir-2025\Hybrid_BE_fit\Human\results\matrix_plots_per_participant\6363354_update_matrices.png
Saved plot: D:\s

: 